# VTuber Builder

## ① 환경 설치

In [ ]:
import pathlib, shutil, subprocess, sys, runpy

REPO_URL = "https://github.com/jujumelona/Virtual-pipeline.git"
REPO_DIR = pathlib.Path("/content/Virtual-pipeline")

def run(cmd, timeout):
    print(f"[Colab 준비] $ {' '.join(cmd)}", flush=True)
    # No capture_output: git, pip, model downloads and errors remain visible.
    subprocess.run(cmd, check=True, text=True, timeout=timeout)

print("①-1 / GitHub main 다운로드 및 동기화", flush=True)
if (REPO_DIR / ".git").is_dir():
    run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL], 60)
    run(["git", "-C", str(REPO_DIR), "fetch", "--prune", "origin", "main"], 180)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    run(["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(REPO_DIR)], 300)

run(["git", "-C", str(REPO_DIR), "checkout", "-B", "main", "origin/main"], 60)
run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], 60)
print("①-2 / Gradio UI 패키지 설치", flush=True)
run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
     "gradio==6.3.0"], 600)

print("①-3 / 환경 준비", flush=True)
# Install in a CHILD Python process. Pip changes installed packages during
# preparation; the notebook kernel must not retain stale Gradio/NumPy imports.
setup_python = (
    "import pathlib, runpy; "
    "REPO_DIR = pathlib.Path('/content/Virtual-pipeline'); "
    "app = runpy.run_path(str(REPO_DIR / 'tools' / 'colab_app.py'), "
    "run_name='vtuber_prepare'); "
    "head, logs = app['ensure_runtime'](); "
    "print('\\n'.join(logs), flush=True); "
    "print('✅ ① 준비 완료. main=' + head[:12], flush=True)"
)
run([sys.executable, "-u", "-c", setup_python], 9000)
print("✅ 환경 설치 완료", flush=True)


## ② 모델 다운로드·검증

In [ ]:
import pathlib, subprocess, sys

repo = pathlib.Path("/content/Virtual-pipeline")
code = (
    "import pathlib, runpy; "
    "repo = pathlib.Path('/content/Virtual-pipeline'); "
    "app = runpy.run_path(str(repo / 'tools' / 'colab_app.py'), run_name='prepare'); "
    "app['prepare_models']()"
)
subprocess.run([sys.executable, "-u", "-c", code], check=True, timeout=3600)
print("✅ 모델 준비 완료", flush=True)


## ③ 생성 UI

In [ ]:
import pathlib, runpy, sys

repo = pathlib.Path("/content/Virtual-pipeline")
# runpy does not put the checkout on sys.path; setup ran in a child process.
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
runpy.run_path(str(repo / "tools" / "colab_app.py"), run_name="__main__")
